In [13]:
# CELL 1: Imports
import pandas as pd
import numpy as np
import json
import random
from collections import defaultdict
import re  # Crucial for text cleaning
from tqdm import tqdm
from transformers import pipeline
import gc # Garbage Collector to free memory
import os
import glob  


# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
# CELL 2: Load Data
# Make sure the path matches your Kaggle input directory
with open('/kaggle/input/testsmall/small dataset2.txt', 'r', encoding='utf-8') as f:
    raw_data = f.readlines()

# Parse data into a list of dictionaries
data = []
for line in raw_data:
    line = line.strip()
    if not line:
        continue
    parts = line.split('\t')
    if len(parts) == 2:
        text, label = parts
        data.append({'text': text.strip(), 'label': label.strip()})
    else:
        data.append({'text': line, 'label': None})

print(f"Parsed {len(data)} samples")

Parsed 99 samples


In [15]:
# CELL 3: Data Preprocessing (Cleaning & Special Char Removal)
def clean_text(text):
    """
    Standardizes text for LICR by removing special characters and noise.
    
    PRESERVES: 
    - Letters & Numbers
    - Basic punctuation (. , ! ?) used for sentence structure
    - Hyphens (-) and Apostrophes (') crucial for Cebuano (e.g., 'adlaw-adlaw', 'wa'y')
    
    REMOVES:
    - URLs, HTML tags
    - Special symbols (@, #, $, %, *, etc.)
    - Emojis (if any)
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase everything
    text = text.lower()
    
    # 2. Remove URLs and Website links
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Remove User Handles (e.g. @username)
    text = re.sub(r'@\w+', '', text)
    
    # 4. Remove Special Characters BUT keep basic punctuation, hyphens, and apostrophes
    # The regex [^...] means "remove anything that is NOT in this list"
    # We keep: a-z, 0-9, whitespace (\s), and specific punctuation (.,!?-')
    text = re.sub(r'[^a-z0-9\s\.\,\!\?\-\']', '', text)
    
    # 5. Isolate Punctuation
    # Adds a space around . , ! ? so "kaayo!" becomes "kaayo !" 
    # This ensures your dictionary matches "kaayo" correctly.
    text = re.sub(r'([.,!?])', r' \1 ', text)
    
    # 6. Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# --- TEST AREA ---
# Test with a messy Cebuano sentence containing symbols and noise
test_str = "Nindot kaayo!!! #Cebu @user123 (Grabe-ka-init) $$ kwarta? wa'y."
print(f"Original: {test_str}")
print(f"Cleaned:  {clean_text(test_str)}")

# Expected Output: "nindot kaayo ! ! ! grabe-ka-init kwarta ? wa'y ."

Original: Nindot kaayo!!! #Cebu @user123 (Grabe-ka-init) $$ kwarta? wa'y.
Cleaned:  nindot kaayo ! ! ! cebu grabe-ka-init kwarta ? wa'y .


In [16]:
# CELL 4: LICR Dictionaries (Optimized & Expanded)
ORTHOGRAPHIC_VARIANTS = {
    # --- PARTICLES & CONJUNCTIONS (Very High Frequency) ---
    'ug': ['og'],
    'ang': ['ahng'],
    'nga': ['ng', 'nnga', 'ngaah'],
    'mga': ['manga', 'mgha', 'mgaa'],
    'kay': ['kai', 'ky', 'kei'],
    'pero': [ 'piro', 'peru'],
    'para': ['pra', 'prah'],
    'bisan': ['bisag', 'bsan', 'maskin'],
    'man': ['man gud', 'man diay', 'mn'],
    'aron': ['arun'],

    # --- NEGATION & FUNCTION WORDS ---
    'wala': ["wa'la", 'wla', 'wa', 'ala', 'waz'],
    'walay': ["wala'y", 'wlay', 'walai', "walay'", 'way', 'wa nay'],
    'dili': ['dli', 'di', "di'li", 'dlee', 'dely'],
    'ayaw': ['yaw', 'ayw'],
    'ambot': ['ambt', 'bot'],

    # --- INTENSIFIERS / ADVERBS ---
    'kaayo': ['kaajo', 'kaayu', 'kayo', 'kaau', 'au'],
    'maayo': ['maajo', 'mayo', 'maayu', 'mau'],
    'gamay': ['gamy', 'gamai', "gamay'", 'gmay'],
    'daghan': ['daghn', 'dagan', 'daghang', "dagha'n", 'dghan'],
    'gyud': ['jud', 'gd', 'gyd', 'gud', 'gayud'],
    'lagi': ['lge', 'lage', 'lgi'],
    'gali': ['galing', 'gli'],
    'karon': ['krn', 'run', 'krun', 'kron'],
    'unya': ['unja', 'nya', 'onja'],
    'kanunay': ['permi', 'kanunai'],
    'dugay': ['dugai', 'dogay', 'dugy'],
    'usahay': ['osahay', 'panagsa'],

    # --- INTERROGATIVES (Boholano/Surigaonon vars included) ---
    'unsa': ["uns'a", 'unssa', 'onsa'],
    'unsaon': ['onsawon', 'unsawn'],
    'asa': ['aha'],
    'ngano': ['ngnu', 'ngno'],
    'kanus-a': ['kansa', 'kanus.a', 'anus-a'],
    'kinsa': ['knsa'],
    'pila': ['pyla'],

    # --- PRONOUNS & POINTERS ---
    'niya': ['nya', 'nia', 'niyaa'],
    'siya': ['sya', 'xa'],
    'sila': ['syla', 'cla', 'ila'],
    'ikaw': ['kaw', 'ikw'],
    'ako': ['ko', 'aq'],
    'kita': [ 'kta'],
    'kana': ['kanha', "kan'a", 'kna'],
    'kani': ['kini', 'ani', 'kni'],
    'kamo': [ 'kmu', 'kamu'],
    'inyo': ['inyu',],
    'naa': ['ania', 'nia'],

    # --- COMMON NOUNS & VERBS ---
    'tao': ['tawo', 'taw'],
    'bata': [ "bat'a", 'bta'],
    'balay': ['balai', 'balae', 'bay'],
    'kauban': ['kuyog', 'kaupan', 'kaobn'],
    'uyab': ['ujab', 'uyb', 'ojab'],
    'salamat': ['slamat', 'lamats'],
    'human': ['humn'],
    'palihug': ['palihog', 'plihug'],
    'kahibalo': ['kabalo', 'kbaw', 'kbalo'],
    'tabang': ['tbang'],
    'kwarta': ['kwrta','kuwarta'],
    'pagkaon': ['pagkawn','kaon'],
    'trabaho': ['trbaho'],

    # --- EMOTIONAL STATES / DESCRIPTORS ---
    'kalipay': ['kalipai', 'kalipaj', 'lipay'],
    'kasakit': ["kasakit'", 'kasakiton'],
    'kahadlok': ['kahadluk', 'kahdlok', 'hadlok'],
    'nindot': ['nindut', 'nendot'],
    'dako': ['dakko', 'daku', 'dakuu'],
    'haom': ['haum', 'hawom'],
    'lami': ['lamii', 'lame', 'lamhi'],
    'kalma': ['kalmaa'],
    'lisod': ['lisud', 'leisod'],
    'guba': ['daut', 'naguba'],
    'mingaw': ['mingawon', 'mngaw'],
    'luoy': ['kaluoy', 'looy', 'kalooy'],
}


# SEMANTIC SUBSTITUTION (Contextual Synonyms)
CEBUANO_SYNONYMS = {
    # --- POSITIVE SENTIMENT ---
    'nindot': ['maayo', 'tsada', 'chada', 'payts', 'swabe'],
    'maayo': ['nindot', 'sakto', 'goods'],
    'lami': ['kalami', 'lami-a'],
    'gwapa': ['guapa', 'gwapaha', 'anyag', 'chix'],
    'gwapo': ['guapo', 'tisoy', 'kaguapo'],
    'hayahay': ['komportable'],
    'lipay': ['malipay', 'malipayon'],
    'kugihan': ['bilib', 'ma-asikaso'],
    'salig': ['kampante', 'tuo'],
    'hinlo': ['limpyo'],
    'bilib': ['saludo'],
    'barato': ['sulit'],
    'buotan': ['bootan', 'maayo', 'boutan'],
    'kusog': ['paspas', 'hataw', 'power'],

    # --- NEGATIVE SENTIMENT ---
    'dautan': ['bati', 'pangit', 'dalo'],
    'hugaw': ['baho', 'hugawan', 'daghab'],
    'binuang': ['kabuang', 'binuangon', 'ilad'],
    'kasuko': ['kalagot', 'lagot', 'pungot'],
    'bati': ['pangit', 'luod', 'way ayo', 'lain'],
    'sapoton': ['mainiton', 'masuko', 'labad'],
    'bogo': ['tanga', 'bugo', 'way alamag'],
    'kapoy': ['luya', 'kapuyan', 'hago'],
    'samok': ['hasol', 'abala', 'gubot'],
    'mahal': ['presyoso', 'overpriced'],
    'hinay': ['kamang', 'loading', 'buntagon'],
    'guba': ['daut', 'naguba', 'broken'],
    'layo': ['layu', 'taliwala'],
}

In [17]:
# CELL 5: Augmentation Logic
def augment_orthographic(text):
    words = text.split()  # Safe to split by space now because of clean_text
    augmented_words = []
    changes = []
    
    for word in words:
        if word in ORTHOGRAPHIC_VARIANTS:
            # chance to change (increased for testing visibility)
            if random.random() < 0.9:
                variant = random.choice(ORTHOGRAPHIC_VARIANTS[word])
                augmented_words.append(variant)
                changes.append(f"{word}->{variant}")
            else:
                augmented_words.append(word)
        else:
            augmented_words.append(word)
    
    return ' '.join(augmented_words), changes

def augment_synonym(text):
    words = text.split()
    augmented_words = []
    changes = []
    
    for word in words:
        if word in CEBUANO_SYNONYMS:
            #  chance to replace
            if random.random() < 0.8:
                synonym = random.choice(CEBUANO_SYNONYMS[word])
                augmented_words.append(synonym)
                changes.append(f"{word}->{synonym}")
            else:
                augmented_words.append(word)
        else:
            augmented_words.append(word)
            
    return ' '.join(augmented_words), changes

In [18]:
# CELL 6: Generate Dataset
augmented_data = []

print("Starting LICR Preprocessing & Augmentation...")

for sample in tqdm(data):
    # 1. CLEANING STEP (Section 3.6.1 of your thesis)
    raw_text = sample['text']
    cleaned_text = clean_text(raw_text)
    
    # Store original (cleaned)
    entry = {
        'original_raw': raw_text,
        'clean_text': cleaned_text,
        # 'label': sample['label'],
        'augmentations': []
    }
    
    # 2. AUGMENTATION STEP (generate 1 orthographic version)
    aug_ortho, changes_ortho = augment_orthographic(cleaned_text)
    
    if changes_ortho: # Only save if changes happened (optional, saves space)
        entry['augmentations'].append({
            'type': 'orthographic',
            'text': aug_ortho,
            'changes': changes_ortho
        })
        
    augmented_data.append(entry)

# Quick Stats
total_aug = sum(len(x['augmentations']) for x in augmented_data)
print(f"\nProcessing complete.")
print(f"Total samples: {len(data)}")
print(f"Total successful augmentations: {total_aug}")

Starting LICR Preprocessing & Augmentation...


100%|██████████| 99/99 [00:00<00:00, 77963.97it/s]


Processing complete.
Total samples: 99
Total successful augmentations: 45


In [19]:
# CELL 7: View Results
for item in augmented_data[:10]:
    print("-" * 50)
    # print(f"Raw:   {item['original_raw']}")
    print(f"Clean: {item['clean_text']}")
    if item['augmentations']:
        for aug in item['augmentations']:
            print(f"Aug ({aug['type']}): {aug['text']}")
            print(f"Changes: {aug['changes']}")
    else:
        print("No augmentation triggered (no matching words found)")

--------------------------------------------------
Clean: ilha ang target audience para sa usa ka fitness club .
Aug (orthographic): ilha ahng target audience pra sa usa ka fitness club .
Changes: ['ang->ahng', 'para->pra']
--------------------------------------------------
Clean: ang mosunod ba nga string usa ka balido nga numero sa telepono o dili output 1 alang sa balido ug 0 alang sa dili balido .
Aug (orthographic): ahng mosunod ba ngaah string usa ka balido nnga numero sa telepono o dli output 1 alang sa balido og 0 alang sa dely balido .
Changes: ['ang->ahng', 'nga->ngaah', 'nga->nnga', 'dili->dli', 'ug->og', 'dili->dely']
--------------------------------------------------
Clean: para sa gihatag nga liriko , sulati ang blangko .
Aug (orthographic): prah sa gihatag ng liriko , sulati ahng blangko .
Changes: ['para->prah', 'nga->ng', 'ang->ahng']
--------------------------------------------------
Clean: i-edit ang mosunod nga sentence aron mas tukma ang pagpanigarilyo makadaot sa 

In [21]:
# CELL 8.5: Define Linguistic Metadata Features (Expanded)

LINGUISTIC_FEATURES = {
    # 1. NEGATION (Reverses sentiment)
    'negation': set([
        'dili', 'di', 'dli', 'dlee', 'dele', 'dely',
        'wala', 'wa', 'wla', 'waa', 'waz', 'ala',
        'walay', 'way', "wa'y", "waa'y",
        'ayaw', 'yaw', 'ayw',
        'ambot', 'bot', 'ambt' 
    ]),

    # 2. CONTRAST (Shifts sentiment mid-sentence)
    'contrast': set([
        'pero', 'pro', 'piro', 'peru',
        'apan', 'kundi',
        'bisan', 'bisag', 'bsan', 'maskin', 'maski', 
        'hinuon', 
        'ugaling',
        'galing', 
        'basta',  
        'bahala', 'bahalag' 
    ]),

    # 3. INTENSIFIER (Strengthens sentiment)
    'intensifier': set([
        # Standard Intensifiers
        'kaayo', 'kaau', 'kayo', 'au', 
        'gyud', 'jud', 'gd', 'gud', 'gayud',
        'lagi', 'lage', 'lge',
        'grabe', 'grabeha', 'grabeng',
        'hastang', 'hasta',
        'pwerteng', 'pwerte', 'puwerte',
        'labihan', 'labi',
        'mas', 'pinaka', 
        'taman', 'sobra', 'todo', 'super',
        'uy', 'oy', 'uie', 
        'ba', 
        'man', 'mn',
        'gani', 'gali', 
        'diay' 
    ]),

    # 4. HEDGE (Weakens sentiment / Uncertainty / Hearsay)
    'hedge': set([
        'murag', 'mura', 'mura ug', 'mura og', 'mrag',
        'siguro', 'cguro', 'seguro', 'gurow', 'guro',
        'tingali', 'tali', 
        'daw', 'raw', 
        'kuno', 'knu',
        'basin', 'basig', 'bacn', 
        'ambot lang', 
        'medyo', 'mejo', 'medjo', 
        'hapit' 
    ])
}

# The function remains the same, but now uses these richer lists
def extract_metadata(text):
    if not isinstance(text, str): return {}
    words = set(text.lower().split()) 
    
    meta = {
        "has_negation": False,
        "has_contrast": False,
        "has_intensifier": False,
        "has_hedge": False
    }
    
    if not words.isdisjoint(LINGUISTIC_FEATURES['negation']):
        meta['has_negation'] = True
    if not words.isdisjoint(LINGUISTIC_FEATURES['contrast']):
        meta['has_contrast'] = True
    if not words.isdisjoint(LINGUISTIC_FEATURES['intensifier']):
        meta['has_intensifier'] = True
    if not words.isdisjoint(LINGUISTIC_FEATURES['hedge']):
        meta['has_hedge'] = True
        
    return meta

In [22]:
# CELL 10: Inspect Only Valid Pairs (The "Consistency" Data)

# Build final_dataset in memory first
final_dataset = []
for sample in data:
    raw_text = sample['text']
    cleaned_text = clean_text(raw_text)
    if not cleaned_text or len(cleaned_text.strip()) < 2:
        continue
    aug_ortho, changes_ortho = augment_orthographic(cleaned_text)
    has_aug = len(changes_ortho) > 0
    meta = extract_metadata(cleaned_text)
    entry = {
        "original": cleaned_text,
        "augmented": aug_ortho if has_aug else None,
        "has_augmentation": has_aug,
        "metadata": meta,
        "changes": changes_ortho if has_aug else []
    }
    final_dataset.append(entry)

# Now inspect valid pairs
print("Displaying the first 10 VALID PAIRS for Consistency Training:")
print("(Skipping samples with no augmentation...)\n")

count = 0
for i, entry in enumerate(final_dataset):
    if entry['has_augmentation']:
        count += 1
        print(f"Pair #{count} (Original Index {i})")
        print(f"  X  (Original):  {entry['original']}")
        print(f"  X' (Augmented): {entry['augmented']}")
        print("-" * 50)
        if count >= 10:
            break

if count == 0:
    print("Warning: No augmented pairs found in the dataset.")

Displaying the first 10 VALID PAIRS for Consistency Training:
(Skipping samples with no augmentation...)

Pair #1 (Original Index 0)
  X  (Original):  ilha ang target audience para sa usa ka fitness club .
  X' (Augmented): ilha ang target audience pra sa usa ka fitness club .
--------------------------------------------------
Pair #2 (Original Index 1)
  X  (Original):  ang mosunod ba nga string usa ka balido nga numero sa telepono o dili output 1 alang sa balido ug 0 alang sa dili balido .
  X' (Augmented): ahng mosunod ba nga string usa ka balido ng numero sa telepono o di output 1 alang sa balido og 0 alang sa dely balido .
--------------------------------------------------
Pair #3 (Original Index 2)
  X  (Original):  para sa gihatag nga liriko , sulati ang blangko .
  X' (Augmented): prah sa gihatag nga liriko , sulati ahng blangko .
--------------------------------------------------
Pair #4 (Original Index 3)
  X  (Original):  i-edit ang mosunod nga sentence aron mas tukma ang pa

In [23]:
# CELL 11: PRODUCTION RUN 

# --- 1. CONFIGURATION ---
# We can increase the chunk size since AI processing is removed
CHUNK_SIZE = 10000       
OUTPUT_DIR = 'processed_chunks_final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. MAIN PROCESSING LOOP ---
print(f"STARTING FAST RUN on {len(data)} samples...")
print(f"Saving chunks to: {OUTPUT_DIR}/")

current_chunk_data = []
chunk_index = 1
global_counter = 1 # Tracks ID numbers

# Iterate through ALL data
for i, sample in enumerate(tqdm(data)):
    
    # A. PRE-PROCESSING
    raw_text = sample['text']
    cleaned_text = clean_text(raw_text)
    
    # Skip empty/garbage lines
    if not cleaned_text or len(cleaned_text.strip()) < 2:
        continue
    
    # Augment
    aug_ortho, changes_ortho = augment_orthographic(cleaned_text)
    has_aug = len(changes_ortho) > 0
    
    # Metadata
    meta = extract_metadata(cleaned_text)
    
    # Generate ID (e.g., cebu_000001)
    sample_id = f"cebu_{global_counter:06d}"
    global_counter += 1
    
    # Prepare the entry object to MATCH EXACTLY what you requested
    entry = {
        "id": sample_id,
        "original": cleaned_text,
        "augmented": aug_ortho if has_aug else None,
        "has_augmentation": has_aug,
        "metadata": meta,
        "changes": changes_ortho if has_aug else []
    }
    
    current_chunk_data.append(entry)
    
    # B. PROCESS CHUNK (Just save to disk)
    if len(current_chunk_data) >= CHUNK_SIZE:
        filename = f"{OUTPUT_DIR}/licr_part_{chunk_index}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(current_chunk_data, f, ensure_ascii=False, indent=2)
        
        current_chunk_data = [] 
        chunk_index += 1
        gc.collect() 

# C. FINAL LEFTOVERS
if current_chunk_data:
    filename = f"{OUTPUT_DIR}/licr_part_{chunk_index}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(current_chunk_data, f, ensure_ascii=False, indent=2)

print("\nDONE. All samples processed with IDs and metadata.")

STARTING FAST RUN on 99 samples...
Saving chunks to: processed_chunks_final/


100%|██████████| 99/99 [00:00<00:00, 58434.58it/s]


DONE. All samples processed with IDs and metadata.


In [24]:
# CELL 12: MERGE ALL CHUNKS


OUTPUT_DIR = 'processed_chunks_final' 

# 1. Find all the chunk files
all_files = sorted(glob.glob(f"{OUTPUT_DIR}/licr_part_*.json"))
combined_data = []

print(f"Found {len(all_files)} chunk files in {OUTPUT_DIR}. Merging...")

if len(all_files) == 0:
    print("ERROR: No files found! Check if Cell 11 ran successfully.")
else:
    # 2. Combine them
    for file in tqdm(all_files):
        with open(file, 'r', encoding='utf-8') as f:
            chunk = json.load(f)
            combined_data.extend(chunk)

    # 3. Save the Master File
    output_filename = 'LICR_Cebert_Final_Format.json'
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, ensure_ascii=False, indent=2)

    print(f"COMPLETE! Final dataset size: {len(combined_data)}")
    print(f"Saved to: {output_filename}")
    
    # 4. Verify the first entry to ensure the format is perfect
    if len(combined_data) > 0:
        print("\n--- SAMPLE ENTRY CHECK ---")
        print(json.dumps(combined_data[0], indent=2, ensure_ascii=False))

Found 1 chunk files in processed_chunks_final. Merging...


100%|██████████| 1/1 [00:00<00:00, 1109.31it/s]

COMPLETE! Final dataset size: 99
Saved to: LICR_Cebert_Final_Format.json

--- SAMPLE ENTRY CHECK ---
{
  "id": "cebu_000001",
  "original": "ilha ang target audience para sa usa ka fitness club .",
  "augmented": "ilha ahng target audience pra sa usa ka fitness club .",
  "has_augmentation": true,
  "metadata": {
    "has_negation": false,
    "has_contrast": false,
    "has_intensifier": false,
    "has_hedge": false
  },
  "changes": [
    "ang->ahng",
    "para->pra"
  ]
}
